# Module 02, Notebook 1: ITS from Scratch in PyMC

## Learning Objectives
By completing this notebook, you will:
1. Build a complete Bayesian ITS model from scratch using PyMC
2. Verify that your model produces the same results as CausalPy
3. Compute the counterfactual and causal effect manually from posterior samples
4. Add an AR(1) error structure to handle autocorrelation
5. Understand the connection between PyMC code and the mathematical ITS model

## The Value of Building from Scratch

Building the ITS model in raw PyMC:
- Confirms your understanding of what CausalPy is doing internally
- Lets you modify any part of the model (likelihood, priors, error structure)
- Is the only way to implement features not in CausalPy (Poisson, Student-t, AR(1))
- Prepares you to debug and extend for your own research

## Estimated Time: 15 minutes

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pymc as pm
import arviz as az
import causalpy as cp

np.random.seed(42)
plt.style.use("seaborn-v0_8-whitegrid")
%matplotlib inline

print(f"PyMC: {pm.__version__}")
print(f"ArviZ: {az.__version__}")
print(f"CausalPy: {cp.__version__}")

## 1. The Dataset

We use the same smoking ban dataset from Module 01, but this time we build the model entirely in PyMC before replicating it with CausalPy.

In [ ]:
# Recreate the smoking ban dataset (same as Module 01)
np.random.seed(42)
n_months = 48
n_pre = 24
intervention_month = n_pre

months = np.arange(n_months).astype(float)
treated = (months >= intervention_month).astype(float)
t_post = np.maximum(months - intervention_month, 0).astype(float)

# True parameters
true_alpha = 85.0
true_beta1 = -0.15
true_beta2 = -12.0  # Level change
true_beta3 = -0.10  # Slope change
true_sigma = 4.0

ami_rate = (
    true_alpha
    + true_beta1 * months
    + true_beta2 * treated
    + true_beta3 * t_post * treated
    + np.random.normal(0, true_sigma, n_months)
)

df = pd.DataFrame({
    "month": months,
    "ami_rate": ami_rate,
    "treated": treated,
    "t_post": t_post,
})

# Pre-intervention statistics (for prior scaling)
y_pre_mean = df.loc[df["treated"] == 0, "ami_rate"].mean()
y_pre_std = df.loc[df["treated"] == 0, "ami_rate"].std()

print(f"Pre-intervention mean: {y_pre_mean:.1f}")
print(f"Pre-intervention std: {y_pre_std:.1f}")
print(f"n_pre: {n_pre}, n_post: {n_months - n_pre}")

## 2. Build the ITS Model in Raw PyMC

We implement the full segmented regression:
$$Y_t \sim \mathcal{N}(\alpha + \beta_1 t + \beta_2 D_t + \beta_3 (t-t^*)D_t, \sigma^2)$$

In [ ]:
# Build the design matrix manually
# X = [intercept, month, treated, t_post]

X = np.column_stack([
    np.ones(n_months),    # Intercept
    months,               # Time trend
    treated,              # Level change
    t_post,               # Slope change
])

y = ami_rate

print(f"Design matrix X shape: {X.shape}")
print("Column order: [Intercept, month, treated, t_post]")
print("First few rows:")
print(pd.DataFrame(X, columns=["Intercept", "month", "treated", "t_post"]).head())
print("\nRows around the intervention:")
print(pd.DataFrame(X, columns=["Intercept", "month", "treated", "t_post"]).iloc[22:27])

In [ ]:
# Build the complete ITS model in PyMC
# This is exactly what CausalPy's LinearRegression does internally

n_features = X.shape[1]
col_names = ["Intercept", "month", "treated", "t_post"]

with pm.Model() as its_pymc_model:

    # ---- DATA ----
    # pm.Data allows updating for counterfactual predictions
    X_data = pm.Data("X", X, mutable=True)
    y_data = pm.Data("y_obs", y, mutable=True)

    # ---- PRIORS ----
    # Weakly informative priors scaled to the data
    # Each coefficient gets its own named variable for interpretability

    # Intercept: centered near the pre-period mean
    alpha = pm.Normal("Intercept", mu=y_pre_mean, sigma=2 * y_pre_std)

    # Pre-intervention slope: small relative to outcome scale
    beta_month = pm.Normal("month", mu=0, sigma=y_pre_std * 0.1)

    # Level change: within ±2 standard deviations
    beta_treated = pm.Normal("treated", mu=0, sigma=y_pre_std)

    # Slope change: typically smaller than level change
    beta_t_post = pm.Normal("t_post", mu=0, sigma=y_pre_std * 0.1)

    # Noise standard deviation
    sigma = pm.HalfNormal("sigma", sigma=y_pre_std)

    # ---- LINEAR PREDICTOR ----
    # Deterministic node: mu[t] = alpha + beta_month*t + beta_treated*D_t + beta_t_post*(t-t*)*D_t
    mu = pm.Deterministic(
        "mu",
        alpha + beta_month * X_data[:, 1] + beta_treated * X_data[:, 2] + beta_t_post * X_data[:, 3]
    )

    # ---- LIKELIHOOD ----
    y_hat = pm.Normal("y_hat", mu=mu, sigma=sigma, observed=y_data)

print("Model built successfully.")
print(f"Free variables: {[v.name for v in its_pymc_model.free_RVs]}")

In [ ]:
# Sample from the posterior using NUTS

with its_pymc_model:
    idata_scratch = pm.sample(
        draws=1000,
        tune=1000,
        chains=4,
        target_accept=0.9,
        progressbar=True,
        random_seed=42,
    )

print("\nSampling complete.")
print(f"Posterior shape for 'treated': {idata_scratch.posterior['treated'].shape}")

In [ ]:
# Extract and print the causal effect estimates

posterior = idata_scratch.posterior

print("ITS From-Scratch Model: Causal Effect Estimates")
print("=" * 55)

for var, true_val in [
    ("Intercept", true_alpha),
    ("month", true_beta1),
    ("treated", true_beta2),
    ("t_post", true_beta3),
    ("sigma", true_sigma),
]:
    samples = posterior[var].values.flatten()
    mean = samples.mean()
    hdi = az.hdi(samples, hdi_prob=0.94)
    print(f"  {var:15s}: mean={mean:+7.3f}  94% HDI=[{hdi[0]:+7.3f}, {hdi[1]:+7.3f}]  true={true_val:+7.3f}")

## 3. Compare with CausalPy

Now fit the same model using CausalPy and verify the estimates match.

In [ ]:
# Fit the identical model using CausalPy
# The formula and data structure match exactly

result_causalpy = cp.InterruptedTimeSeries(
    data=df,
    treatment_time=intervention_month,
    formula="ami_rate ~ 1 + month + treated + t_post",
    model=cp.pymc_models.LinearRegression(
        sample_kwargs={
            "draws": 1000,
            "tune": 1000,
            "chains": 4,
            "target_accept": 0.9,
            "progressbar": True,
            "random_seed": 42,
        }
    ),
)

print("CausalPy model fitted.")

In [ ]:
# Compare the key causal parameters between the two models

print("COMPARISON: From-Scratch PyMC vs CausalPy")
print("=" * 60)
print(f"{'Parameter':<15} {'From-Scratch':>15} {'CausalPy':>15} {'Diff':>10}")
print("-" * 60)

for var in ["treated", "t_post"]:
    scratch_mean = idata_scratch.posterior[var].values.flatten().mean()
    causalpy_mean = result_causalpy.idata.posterior[var].values.flatten().mean()
    diff = abs(scratch_mean - causalpy_mean)
    print(f"  {var:<13} {scratch_mean:>+15.3f} {causalpy_mean:>+15.3f} {diff:>10.4f}")

print()
print("Note: Small differences are expected due to random sampling (MCMC).")
print("Differences < 0.1 in posterior means indicate the models are equivalent.")

## 4. Manual Counterfactual Computation

Compute the counterfactual trajectory manually from posterior samples, without using CausalPy's built-in visualization.

In [ ]:
# Compute the counterfactual manually from posterior samples
# Counterfactual Y(0) = alpha + beta_month * t
# (treatment variables set to zero: treated=0, t_post=0)

posterior = idata_scratch.posterior

alpha_samples = posterior["Intercept"].values.flatten()
beta1_samples = posterior["month"].values.flatten()
beta2_samples = posterior["treated"].values.flatten()
beta3_samples = posterior["t_post"].values.flatten()

# Posterior predictive mean trajectory (at observed X)
fitted_mean = np.outer(alpha_samples, np.ones(n_months)) + np.outer(beta1_samples, months)
# Add treatment effect for post-intervention
for i in range(len(alpha_samples)):
    fitted_mean[i] += beta2_samples[i] * treated + beta3_samples[i] * t_post

# Counterfactual: only the pre-trend (no treatment effect)
counterfactual = np.outer(alpha_samples, np.ones(n_months)) + np.outer(beta1_samples, months)

# Point estimates
cf_mean = counterfactual.mean(axis=0)
cf_lower = np.percentile(counterfactual, 3, axis=0)
cf_upper = np.percentile(counterfactual, 97, axis=0)

fitted_mean_pt = fitted_mean.mean(axis=0)

# Causal effect at each post-intervention time
causal_effect_samples = fitted_mean[:, n_pre:] - counterfactual[:, n_pre:]

print(f"Average causal effect (post-intervention): {causal_effect_samples.mean():.2f}")
print(f"True average effect: {true_beta2 + true_beta3 * np.arange(n_months - n_pre).mean():.2f}")

In [ ]:
# Visualize the from-scratch ITS results

fig, axes = plt.subplots(2, 1, figsize=(13, 9))

# Top panel: observed data, fitted line, and counterfactual
ax = axes[0]
ax.scatter(months[:n_pre], ami_rate[:n_pre], s=25, color="#2c3e50", zorder=5, label="Observed (pre)")
ax.scatter(months[n_pre:], ami_rate[n_pre:], s=25, color="#e74c3c", zorder=5, label="Observed (post)")

# Fitted model (with uncertainty)
ax.plot(months, fitted_mean_pt, color="#3498db", linewidth=2, label="Fitted (posterior mean)")
fit_lower = np.percentile(fitted_mean, 3, axis=0)
fit_upper = np.percentile(fitted_mean, 97, axis=0)
ax.fill_between(months, fit_lower, fit_upper, alpha=0.2, color="#3498db")

# Counterfactual (post-intervention only)
ax.plot(months[n_pre:], cf_mean[n_pre:], "--", color="#27ae60", linewidth=2, label="Counterfactual (posterior mean)")
ax.fill_between(months[n_pre:], cf_lower[n_pre:], cf_upper[n_pre:], alpha=0.15, color="#27ae60")

ax.axvline(intervention_month, color="black", linestyle=":", linewidth=2)
ax.set_title("ITS Built from Scratch in PyMC: AMI Rate vs Counterfactual", fontsize=13)
ax.set_ylabel("AMI Rate per 100,000")
ax.legend(fontsize=10)

# Bottom panel: causal effect
ax2 = axes[1]
causal_mean = causal_effect_samples.mean(axis=0)
causal_lower = np.percentile(causal_effect_samples, 3, axis=0)
causal_upper = np.percentile(causal_effect_samples, 97, axis=0)
post_months = months[n_pre:]

ax2.plot(post_months, causal_mean, color="#e74c3c", linewidth=2, label="Estimated causal effect")
ax2.fill_between(post_months, causal_lower, causal_upper, alpha=0.3, color="#e74c3c")
ax2.axhline(0, color="black", linestyle="--", linewidth=1.5)
ax2.set_ylabel("Causal Effect (AMI/100k)")
ax2.set_xlabel("Month")
ax2.set_title("Posterior Distribution of Causal Effect Over Time", fontsize=13)
ax2.legend(fontsize=10)

plt.tight_layout()
plt.show()

## 5. Add AR(1) Error Structure (Extension)

CausalPy does not have built-in AR(1) error support. Here we extend the model to handle autocorrelated errors. This is the main advantage of building in raw PyMC.

In [ ]:
# ITS model with AR(1) error structure
# Useful when residuals from the basic model show autocorrelation

with pm.Model() as its_ar1_model:

    # ---- PRIORS (same as before) ----
    alpha = pm.Normal("Intercept", mu=y_pre_mean, sigma=2 * y_pre_std)
    beta_month = pm.Normal("month", mu=0, sigma=y_pre_std * 0.1)
    beta_treated = pm.Normal("treated", mu=0, sigma=y_pre_std)
    beta_t_post = pm.Normal("t_post", mu=0, sigma=y_pre_std * 0.1)

    # AR(1) autocorrelation parameter
    # Prior: Uniform(-1, 1) — allows any stationary AR(1) process
    rho = pm.Uniform("rho", lower=-1, upper=1)

    # Innovation standard deviation
    sigma = pm.HalfNormal("sigma", sigma=y_pre_std)

    # ---- LINEAR PREDICTOR ----
    mu_det = pm.Deterministic(
        "mu_det",
        alpha + beta_month * months + beta_treated * treated + beta_t_post * t_post
    )

    # ---- AR(1) ERRORS ----
    # eps_t = rho * eps_{t-1} + u_t, where u_t ~ N(0, sigma^2)
    # PyMC's AR distribution handles this
    ar_errors = pm.AR(
        "ar_errors",
        rho=[rho],      # AR(1) coefficient
        sigma=sigma,    # Innovation std dev
        shape=n_months,
    )

    # ---- COMBINED MEAN ----
    mu = pm.Deterministic("mu", mu_det + ar_errors)

    # ---- LIKELIHOOD ----
    # Using small sigma_obs since errors are already in mu
    y_hat = pm.Normal("y_hat", mu=mu, sigma=0.01, observed=ami_rate)

# Sample
with its_ar1_model:
    idata_ar1 = pm.sample(
        draws=1000,
        tune=1500,
        chains=4,
        target_accept=0.95,
        progressbar=True,
        random_seed=42,
    )

print("AR(1) model sampling complete.")

In [ ]:
# Compare basic model vs AR(1) model

print("Comparison: Basic ITS vs AR(1) ITS")
print("=" * 60)
print(f"{'Parameter':<15} {'Basic':>15} {'AR(1)':>15} {'True':>10}")
print("-" * 60)

for var, true_val in [("treated", true_beta2), ("t_post", true_beta3)]:
    basic_mean = idata_scratch.posterior[var].values.flatten().mean()
    ar1_mean = idata_ar1.posterior[var].values.flatten().mean()
    basic_std = idata_scratch.posterior[var].values.flatten().std()
    ar1_std = idata_ar1.posterior[var].values.flatten().std()
    print(f"  {var:<13} {basic_mean:>+15.3f} {ar1_mean:>+15.3f} {true_val:>+10.3f}")
    print(f"  {'(std)':13} {basic_std:>15.3f} {ar1_std:>15.3f}")

# Show estimated AR(1) coefficient
rho_posterior = idata_ar1.posterior["rho"].values.flatten()
print(f"\nEstimated AR(1) coefficient (rho):")
print(f"  Posterior mean: {rho_posterior.mean():.3f}")
print(f"  94% HDI: {az.hdi(rho_posterior, hdi_prob=0.94)}")

# If rho ≈ 0, there is no meaningful autocorrelation
# If rho is confidently different from 0, autocorrelation matters

## Summary

### What You Built
1. A complete ITS model from scratch in PyMC — design matrix, priors, likelihood
2. Manual counterfactual computation from posterior samples
3. An AR(1) extension that handles autocorrelated errors

### Key Insight: CausalPy = Convenience, PyMC = Flexibility
- CausalPy gives you a clean API for standard ITS analyses
- PyMC gives you full control for custom models
- The two are equivalent for standard cases — verified by the comparison in Section 3

### When to Use Raw PyMC
- Count outcomes (Poisson or Negative Binomial likelihood)
- Autocorrelated errors (AR(1) or AR(2) structure)
- Hierarchical ITS across multiple treated units
- Highly informative priors from meta-analyses
- Any case where CausalPy's defaults are not appropriate

### What's Next
**Notebook 2:** Prior sensitivity analysis — systematically comparing how different prior specifications affect the causal effect estimate.